In [ ]:

from prototype_updated_phase2 import load_hivrc
DATA_PATH = "/DATA/WGS_study/YSL/projects/latentgee/examples"

In [39]:
from pathlib import Path
import pandas as pd
import numpy as np
def rm_zero(otu_table, axis=0, cutoff=0.05):
    """
    Prevalence-based filtering for Sample x OTU table.

    Parameters
    ----------
    otu_table : pandas.DataFrame
        Sample x OTU 형태의 데이터프레임
    axis : int, default=0
        0: OTU(column) filtering
        1: Sample(row) filtering
    cutoff : float, default=0.05
        non-zero prevalence cutoff

    Returns
    -------
    pandas.DataFrame
        filtered table
    """
    n_sample = otu_table.shape[0]
    n_otu = otu_table.shape[1]

    if axis == 0:
        nonzero_prevalence = (otu_table > 0).sum(axis=0) / n_sample
        otu_table_filtered = otu_table.loc[:, nonzero_prevalence >= cutoff]
    elif axis == 1:
        nonzero_prevalence = (otu_table > 0).sum(axis=1) / n_otu
        otu_table_filtered = otu_table.loc[nonzero_prevalence >= cutoff, :]
    else:
        raise ValueError("axis must be 0 (OTU filtering) or 1 (sample filtering)")

    return otu_table_filtered




In [58]:
def hivrc_preprocess():
    """
    1) OTU -> genus aggregation
    2) metadata와 샘플 맞춤
    3) HIV, age, gender, Study complete-case만 유지
    4) all-zero genera 제거
    5) 최종 sample x genus matrix와 metadata 반환

    returns
    ---
    genus_table: pd.DataFrame
        sample x genus count matrix
    meta_final : pd.DataFrame
        정렬 완료된 metadata
    """
    sample_id_col="SeqID"
    study_col="Study"
    hiv_col="hivstatus"
    age_col="Age"
    gender_col="gender"
    taxonomy_col="taxonomy"

    
    # -------------------- 1. read count ---------------------
    print("step 1. read count")
    file_path="/DATA/WGS_study/YSL/projects/Data"
    otu_raw_path = Path(f"{file_path}/insight.merged_otus.txt")
    meta_raw_path = Path(f"{file_path}/SupplementaryMaterial.xlsx")

    otu_raw = pd.read_csv(otu_raw_path, sep="\t", encoding = "utf-8", index_col='Resphera Insight (Raw Counts)')
    meta_raw = pd.read_excel(meta_raw_path, header = 1, usecols = "B:L")
    meta_raw.index = meta_raw[sample_id_col]
    otu_raw.index.name = sample_id_col

    otu_raw.reset_index(inplace=True)
    otu_raw.index=otu_raw[sample_id_col]

    # -------------------- 2. OTU -> genus aggregation ---------------------
    print("step 2. OTU -> genus aggregation")
    # taxonomy parsing
    genus = (
        otu_raw[taxonomy_col]
        .astype(str)
        .str.extract(r"g__([^;]*)", expand=False)
        .fillna("")
        .str.strip()
    )
    genus = genus.str.strip()
    genus_lower = genus.str.lower()

    genus = genus.mask(genus_lower.isin([
        "", "uncultured", "unassigned", "ambiguous_taxa", "incertae_sedis"
    ]), "Unassigned")

    genus = genus.replace({
        "uncultured": "Unassigned",
        "unassigned": "Unassigned",
        "Ambiguous_taxa": "Unassigned",
        "Incertae_Sedis": "Unassigned"
    })
    # -------------------- 3. sample columns 추출 ----------------------
    print("step 3. sample columns 추출")
    sample_cols = [col for col in otu_raw.columns if col not in [taxonomy_col, sample_id_col]]
    otu_counts = otu_raw[sample_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    otu_counts["Genus"] = genus.values

    # -------------------- 4. genus aggregration ----------------------
    #    현재: OTU x Sample
    #    집계 후: Genus x Sample
    # ---------------------------
    print("step 4. genus aggregation")
    genus_by_sample = otu_counts.groupby("Genus", dropna=False).sum()

    # sample x genus로 전치
    sample_by_genus = genus_by_sample.T


    # -------------------- 5. metadata와 공통 샘플만 유지 ----------------------
    print("step 5. metadata와 공통 샘플만 유지")
    meta = meta_raw.copy()
    meta[sample_id_col] = meta[sample_id_col].astype(str)
    sample_by_genus.index = sample_by_genus.index.astype(str)

    common_samples = sample_by_genus.index.intersection(meta[sample_id_col])
    sample_by_genus = sample_by_genus.loc[common_samples].copy()
    meta = meta.loc[common_samples] 
    # -------------------- 6. complete-case filtering
    # HIV, age, gender, Study 모두 NA 아닌 샘플만 유지

    print("step 6. complete-case filtering")
    meta[age_col] = pd.to_numeric(meta[age_col], errors="coerce")
    meta[study_col] = pd.Categorical(meta[study_col])    
    keep = meta[[study_col, hiv_col, age_col, gender_col]].notna().all(axis=1)
    meta = meta.loc[keep].copy()
 
    sample_by_genus = sample_by_genus.loc[meta[sample_id_col]].copy()

    # ---------------------------
    # 7. all-zero genera 제거
    #    논문: zero reads across all samples 제거
    # ---------------------------
    print("step 7. all-zero genera 제거")
    sample_by_genus = sample_by_genus.loc[:, sample_by_genus.sum(axis=0) > 0].copy()
    
    assert sample_by_genus.shape[0] == meta.shape[0]
    assert all(sample_by_genus.index == meta[sample_id_col].astype(str))
    assert meta[[study_col, hiv_col, age_col, gender_col]].notna().all().all()

    # ---------------------------
    # 8. 결과 정리
    # ---------------------------
    meta = meta.reset_index(drop=True)
    sample_by_genus.index = meta[sample_id_col].values

    # 최종 요약
    print("Final shape")
    print(f"  samples : {sample_by_genus.shape[0]}")
    print(f"  genera  : {sample_by_genus.shape[1]}")
    print(f"  studies : {meta[study_col].nunique()}")
    print("\nStudy counts")
    print(meta[study_col].value_counts().sort_index())
    
    return sample_by_genus, meta, meta[study_col]

genus_table, meta_final, study_labels = hivrc_preprocess()


step 1. read count
step 2. OTU -> genus aggregation
step 3. sample columns 추출
step 4. genus aggregation
step 5. metadata와 공통 샘플만 유지
step 6. complete-case filtering
step 7. all-zero genera 제거
Final shape
  samples : 936
  genera  : 668
  studies : 14

Study counts
Study
Dillon                  32
Dinh                    36
Dubourg                 56
Lozupone 2013           37
Monaco                 106
Mutlu                    0
Noguera-Julian         232
Nowak2015                0
Nowak2017              119
Perez-Santiago           0
Pinto-Cardoso           43
Serrano-Villar 2016     39
Serrano-Villar 2017     23
Vesterbacka             62
Villanueva-Millan       69
Villar-Garcia           24
Yu                      58
Name: count, dtype: int64


In [79]:
sample_filtering = rm_zero(genus_table, axis=1, cutoff=0.01)
genus_filtering = rm_zero(sample_filtering, axis=0, cutoff=0.01)

In [80]:
genus_filtering

Genus,Abiotrophia,Acetanaerobacterium,Acetitomaculum,Acetivibrio,Acetoanaerobium,Acetobacter,Acidaminococcus,Acinetobacter,Actinobacillus,Actinobaculum,...,Unassigned,Ureaplasma,Vampirovibrio,Varibaculum,Variovorax,Veillonella,Victivallis,Vulcanibacillus,Weissella,Zimmermannella
4EP1,0,258,0,0,0,0,897,0,0,0,...,1045,0,1,0,0,0,87,0,0,0
4EP10,0,0,0,0,0,0,1032,0,0,0,...,134,0,1,0,0,37,0,0,0,0
4EP12,0,114,0,0,0,0,381,0,0,0,...,1306,0,23,0,0,0,31,0,2,0
4EP15,0,32,0,0,0,0,3,2,0,0,...,476,0,164,0,0,0,41,0,0,0
4EP17,0,5,1,0,0,0,1,0,0,0,...,134,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
rgn.9.13,0,0,0,0,0,0,0,0,0,0,...,179,0,0,0,1,5,0,0,6,0
rgn.9.2,0,0,0,0,0,0,0,0,0,0,...,79,2,0,2,0,1,0,0,0,0
rgn.9.3,0,1,0,0,0,0,0,0,0,0,...,523,0,5,0,0,59,0,0,2,0
rgn.9.6,0,2,0,0,0,0,0,0,0,0,...,228,0,0,0,0,0,0,0,0,0
